# Nhom 3 huan luyen baseline PaySim tren Kaggle
ML offline bo sung, khong thay the ho so Spark va ingestion Buoi 6.
Import notebook nay, chon CPU/None, gan ealaxi/paysim1. Khong can upload lai CSV.
Neu gan saved Output notebook Checkpoint, dien SOURCE_OVERRIDE toi raw CSV full.
Dien EXPECTED_SHA256 bang hash manifest checkpoint de xac nhan cung snapshot.
Ngay 4-5: RUN_FINAL_TEST=False, chon mo hinh bang validation.
Ngay 6: khoa code/thiet ke, doi True de cham test. Chay lai la kiem tra tai lap,
khong phai test doc lap moi. Neu da dung test cu de dieu chinh, phai cong khai.
Notebook chua chay tren tai khoan Kaggle, khong co ket qua duoc dien san.


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,csv,hashlib,platform,importlib.util,zipfile
from datetime import datetime,timezone
EXPERIMENT='safe_v1'
SOURCE_OVERRIDE='' # raw full CSV from saved Checkpoint Outputs if attached
EXPECTED_SHA256='' # paste checkpoint manifest hash; recommended
RUN_FINAL_TEST=False
ROOT=Path('/kaggle/working/nhom3_ml')/EXPERIMENT
ROOT.mkdir(parents=True,exist_ok=True)
(ROOT/'src').mkdir(exist_ok=True)
MODEL_OUT=ROOT/'results'
MODEL_OUT.mkdir(exist_ok=True)
if (MODEL_OUT/'test_metrics.json').exists() and not RUN_FINAL_TEST:
    raise RuntimeError('Old test results exist. Use a new EXPERIMENT for a new design; do not mix old test files with a validation-only run.')


## 1 Materialize source code
Ma train duoc nhung san. Khong can upload .py. Neu thieu dependency, xem ModuleNotFoundError, cai thu vien thieu trong notebook roi restart session. CPU du cho Logistic Regression va Random Forest nay.

In [ ]:
FILES={'paysim_experiment.py': '"""Reproducible, local PaySim profile and offline baseline (not Spark serving).\n\nRaw CSV is read only. All preprocessing is label-free; model/threshold selection\nuses validation only. Full test scoring occurs after selecting one ML model.\n"""\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport sys\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nFIELDS = [\n    "step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",\n    "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud", "isFlaggedFraud",\n]\nDTYPES = {"step": "int16", "type": "category", "nameOrig": str, "nameDest": str,\n          "isFraud": "uint8", "isFlaggedFraud": "uint8"}\nDTYPES.update({c: "float64" for c in ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]})\n\n\ndef log(message):\n    print(f"[{time.strftime(\'%H:%M:%S\')}] {message}", flush=True)\n\n\ndef save_json(path, value):\n    with path.open("x", encoding="utf-8") as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)\n        stream.write("\\n")\n\n\ndef profile(source, audit, out):\n    if (out / "profile.json").exists():\n        raise FileExistsError("profile.json exists; use a new output directory")\n    hashes = []\n    step_groups = []\n    zero_amount = {"0": 0, "1": 0}\n    log("Profile: hashing original columns in chunks; raw data will not be modified")\n    for part_num, chunk in enumerate(pd.read_csv(source, dtype=DTYPES, chunksize=400_000), 1):\n        hashes.append(pd.util.hash_pandas_object(chunk[FIELDS], index=False).to_numpy())\n        step_groups.append(chunk.groupby("step", observed=True).agg(rows=("isFraud", "size"), fraud=("isFraud", "sum")))\n        for label in (0, 1):\n            zero_amount[str(label)] += int(((chunk["amount"] == 0) & (chunk["isFraud"] == label)).sum())\n        log(f"Profile: chunk {part_num}, rows {sum(len(x) for x in hashes):,}")\n    all_hashes = np.concatenate(hashes)\n    del hashes\n    original_order = all_hashes.copy()\n    all_hashes.sort()\n    duplicate_hashes = np.unique(all_hashes[1:][all_hashes[1:] == all_hashes[:-1]])\n    del all_hashes\n    duplicate_extra = 0\n    duplicate_involved = 0\n    duplicate_by_label = {"0": 0, "1": 0}\n    if duplicate_hashes.size:\n        # Hashes only find candidates. Confirm equality on ALL original columns.\n        candidates = []\n        cursor = 0\n        for chunk in pd.read_csv(source, dtype=DTYPES, chunksize=400_000):\n            mask = np.isin(original_order[cursor:cursor + len(chunk)], duplicate_hashes)\n            if mask.any():\n                candidates.append(chunk.loc[mask, FIELDS])\n            cursor += len(chunk)\n        possible = pd.concat(candidates, ignore_index=True)\n        extra = possible.duplicated(subset=FIELDS, keep="first")\n        involved = possible.duplicated(subset=FIELDS, keep=False)\n        duplicate_extra = int(extra.sum())\n        duplicate_involved = int(involved.sum())\n        for label in (0, 1):\n            duplicate_by_label[str(label)] = int((extra & (possible["isFraud"] == label)).sum())\n    del original_order\n    hourly = pd.concat(step_groups).groupby(level=0).sum().sort_index()\n    hourly["legitimate"] = hourly["rows"] - hourly["fraud"]\n    hourly["fraud_rate"] = hourly["fraud"] / hourly["rows"]\n    cumulative = hourly["rows"].cumsum()\n    total = int(hourly["rows"].sum())\n    if total != audit["rows"]:\n        raise ValueError("Profile row count disagrees with independent audit")\n    train_end = int(cumulative.index[np.searchsorted(cumulative.to_numpy(), 0.6 * total)])\n    val_end = int(cumulative.index[np.searchsorted(cumulative.to_numpy(), 0.8 * total)])\n    split_ranges = {\n        "train": (int(hourly.index.min()), train_end),\n        "validation": (train_end + 1, val_end),\n        "test": (val_end + 1, int(hourly.index.max())),\n    }\n    split_info = {}\n    for name, (start, end) in split_ranges.items():\n        group = hourly.loc[start:end]\n        n = int(group["rows"].sum())\n        fraud = int(group["fraud"].sum())\n        if not 0 < fraud < n:\n            raise ValueError(f"Split {name} lacks one class; redesign before fitting")\n        split_info[name] = {"min_step": start, "max_step": end, "rows": n,\n                            "fraud": fraud, "legitimate": n - fraud, "fraud_rate": fraud / n}\n    result = {\n        "source": str(source), "source_sha256": audit["sha256"], "rows": total,\n        "exact_duplicate_extra_rows": duplicate_extra,\n        "exact_duplicate_rows_involved": duplicate_involved,\n        "exact_duplicate_extra_by_label": duplicate_by_label,\n        "duplicate_method": "64-bit hash candidates; candidate equality verified on all 11 parsed original columns; no rows deleted",\n        "zero_amount_by_label": zero_amount,\n        "split_method": "whole hours; first cumulative-hour boundaries reaching 60% and 80% of rows; labels not used to choose boundaries",\n        "splits": split_info,\n        "raw_modified": False,\n        "cleaning_decision": "No automatic deletion/imputation/clipping. Preserve raw; handle forbidden predictors and class imbalance in model design.",\n    }\n    hourly.to_csv(out / "hourly_profile.csv", index_label="step")\n    save_json(out / "profile.json", result)\n    log(f"Profile done: duplicate extra rows {duplicate_extra}; split ends {train_end}, {val_end}")\n    return result\n\n\ndef make_features(frame, categories):\n    hour = (frame["step"].to_numpy(dtype=np.int32) - 1) % 24\n    matrix = np.empty((len(frame), 3 + len(categories)), dtype=np.float32)\n    matrix[:, 0] = np.log1p(frame["amount"].to_numpy(dtype=np.float64))\n    matrix[:, 1] = np.sin(2 * np.pi * hour / 24)\n    matrix[:, 2] = np.cos(2 * np.pi * hour / 24)\n    for index, category in enumerate(categories, 3):\n        matrix[:, index] = (frame["type"] == category).to_numpy(dtype=np.float32)\n    return matrix\n\n\ndef select_threshold(y, score, budget=0.01):\n    # Whole tied-score groups must be included. The budget is enforced exactly.\n    order = np.argsort(-score, kind="stable")\n    ranked = score[order]\n    positives = np.cumsum(y[order], dtype=np.int64)\n    group_ends = np.r_[np.flatnonzero(ranked[:-1] != ranked[1:]), len(ranked) - 1]\n    allowed = group_ends[(group_ends + 1) <= int(np.floor(len(y) * budget))]\n    if not len(allowed):\n        return float(np.nextafter(np.max(score), np.inf))\n    total_positive = y.sum()\n    best = max(allowed, key=lambda i: (positives[i] / total_positive,\n                                      positives[i] / (i + 1), -(i + 1)))\n    return float(ranked[best])\n\n\ndef metrics(y, score, threshold):\n    from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score\n    predicted = score >= threshold\n    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()\n    precision = float(tp / (tp + fp)) if tp + fp else 0.0\n    recall = float(tp / (tp + fn)) if tp + fn else 0.0\n    return {\n        "rows": len(y), "fraud": int(y.sum()), "prevalence": float(y.mean()),\n        "average_precision": float(average_precision_score(y, score)),\n        "roc_auc_secondary": float(roc_auc_score(y, score)),\n        "threshold": float(threshold), "precision": precision, "recall": recall,\n        "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,\n        "alerts": int(tp + fp), "alert_rate": float((tp + fp) / len(y)),\n        "false_positives_per_1000_legitimate": float(1000 * fp / (fp + tn)),\n        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),\n    }\n\n\ndef train(source, audit, prof, out):\n    from sklearn.ensemble import RandomForestClassifier\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    import joblib\n    from threadpoolctl import threadpool_limits\n    if (out / "baseline_results.json").exists():\n        raise FileExistsError("baseline_results.json exists; use new output directory")\n    log("Loading only permitted columns plus isolated labels/rule baseline")\n    frame = pd.read_csv(source, usecols=["step", "type", "amount", "isFraud", "isFlaggedFraud"], dtype=DTYPES)\n    masks = {name: frame["step"].between(spec["min_step"], spec["max_step"]).to_numpy()\n             for name, spec in prof["splits"].items()}\n    categories = sorted(frame.loc[masks["train"], "type"].unique().tolist())\n    feature_names = ["log1p_amount", "hour_sin", "hour_cos", *[f"type_{c}" for c in categories]]\n    assert not any(c in feature_names for c in ["isFraud", "isFlaggedFraud", "step", "nameOrig", "nameDest"])\n    train_x = make_features(frame.loc[masks["train"]], categories)\n    val_x = make_features(frame.loc[masks["validation"]], categories)\n    train_y = frame.loc[masks["train"], "isFraud"].to_numpy()\n    val_y = frame.loc[masks["validation"], "isFraud"].to_numpy()\n    candidates = [\n        (f"LR_weight_{weight}", make_pipeline(StandardScaler(), LogisticRegression(\n            C=1.0, max_iter=250, class_weight={0: 1.0, 1: float(weight)}, random_state=42)))\n        for weight in (1, 10, 50)\n    ]\n    candidates.append(("RF_50_depth8_balanced", RandomForestClassifier(\n        n_estimators=50, max_depth=8, min_samples_leaf=20, max_samples=0.5,\n        class_weight="balanced_subsample", n_jobs=2, random_state=42)))\n    summaries = []\n    best = None\n    best_key = None\n    with threadpool_limits(limits=2):\n        for name, model in candidates:\n            log(f"Fit {name}: {len(train_y):,} rows, ALL chronological train rows")\n            started = time.perf_counter()\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter("always")\n                model.fit(train_x, train_y)\n            fit_seconds = time.perf_counter() - started\n            score = model.predict_proba(val_x)[:, 1]\n            threshold = select_threshold(val_y, score)\n            evaluation = metrics(val_y, score, threshold)\n            result = {"model": name, "fit_seconds": fit_seconds,\n                      "warnings": [str(w.message) for w in caught], "validation": evaluation}\n            summaries.append(result)\n            log(f"Validation {name}: AP={evaluation[\'average_precision\']:.6f}, recall={evaluation[\'recall\']:.4f}, alerts={evaluation[\'alert_rate\']:.4%}")\n            key = (evaluation["recall"], evaluation["precision"], evaluation["average_precision"])\n            if best_key is None or key > best_key:\n                best_key, best = key, (name, model, threshold)\n        del train_x, val_x\n        selected_name, selected, threshold = best\n        log(f"LOCKED selection before test scoring: {selected_name}, threshold={threshold}")\n        test_frame = frame.loc[masks["test"]]\n        test_x = make_features(test_frame, categories)\n        test_y = test_frame["isFraud"].to_numpy()\n        started = time.perf_counter()\n        test_score = selected.predict_proba(test_x)[:, 1]\n        predict_seconds = time.perf_counter() - started\n        test_result = metrics(test_y, test_score, threshold)\n        rule = ((test_frame["type"] == "TRANSFER") & (test_frame["amount"] > 200_000)).to_numpy(dtype=float)\n        test_baselines = {\n            "always_legitimate": metrics(test_y, np.zeros(len(test_y)), 0.5),\n            "transfer_amount_gt_200000_fixed_rule": metrics(test_y, rule, 0.5),\n            "dataset_isFlaggedFraud_fixed_rule": metrics(test_y, test_frame["isFlaggedFraud"].to_numpy(dtype=float), 0.5),\n        }\n        val_baselines = {}\n        val_frame = frame.loc[masks["validation"]]\n        val_rule = ((val_frame["type"] == "TRANSFER") & (val_frame["amount"] > 200_000)).to_numpy(dtype=float)\n        val_baselines["transfer_amount_gt_200000_fixed_rule"] = metrics(val_y, val_rule, 0.5)\n        val_baselines["dataset_isFlaggedFraud_fixed_rule"] = metrics(val_y, val_frame["isFlaggedFraud"].to_numpy(dtype=float), 0.5)\n    versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__}\n    for package in ["scikit-learn", "scipy", "joblib", "threadpoolctl", "cloudpickle", "narwhals"]:\n        \n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    results = {\n        "source_sha256": audit["sha256"], "versions": versions, "features": feature_names,\n        "excluded_predictors": ["isFraud", "isFlaggedFraud", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest", "nameOrig", "nameDest", "row_number", "event_id"],\n        "splits": prof["splits"], "candidate_validation": summaries,\n        "selection_policy": "maximize validation recall, then precision, then AP, with <=1% validation alerts; all tied scores handled together; test never used for tuning",\n        "validation_fixed_baselines": val_baselines,\n        "selected_model": selected_name, "selected_threshold": threshold,\n        "selected_test": test_result, "test_fixed_baselines": test_baselines,\n        "offline_bulk_predict_seconds": predict_seconds,\n        "offline_bulk_rows_per_second": len(test_y) / predict_seconds,\n        "benchmark_warning": "Local sklearn batch timing ONLY; not Kafka/Spark end-to-end throughput or latency. Test alert rate may exceed validation budget under distribution shift.",\n        "raw_modified": False,\n    }\n    model_artifact = out / "selected_model.joblib"\n    if model_artifact.exists():\n        raise FileExistsError(str(model_artifact))\n    joblib.dump(selected, model_artifact)\n    reloaded = joblib.load(model_artifact)\n    np.testing.assert_allclose(reloaded.predict_proba(test_x[:64])[:, 1], test_score[:64], rtol=1e-7, atol=1e-10)\n    results["saved_model_reload_check"] = "pass, first 64 test scores unchanged"\n    save_json(out / "baseline_results.json", results)\n    save_json(out / "model_manifest.json", {"model": selected_name, "threshold": threshold,\n              "type_categories": categories, "feature_order": feature_names,\n              "feature_version": "safe_v1_logamount_hour_type", "source_sha256": audit["sha256"],\n              "model_library": "sklearn", "raw_step_is_not_predictor": True})\n    with (out / "requirements-lock.txt").open("x", encoding="utf-8") as stream:\n        stream.write("\\n".join(f"{name}=={version}" for name, version in versions.items() if name != "python") + "\\n")\n    log(f"DONE: selected test AP={test_result[\'average_precision\']:.6f}, precision={test_result[\'precision\']:.4f}, recall={test_result[\'recall\']:.4f}")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--csv", type=Path, required=True)\n    parser.add_argument("--audit", type=Path, required=True)\n    parser.add_argument("--out", type=Path, required=True)\n    parser.add_argument("--profile-only", action="store_true")\n    args = parser.parse_args()\n    source = args.csv.resolve(strict=True)\n    audit = json.loads(args.audit.read_text(encoding="utf-8"))\n    if audit["essential_errors"]:\n        raise ValueError("Audit found essential errors. Review before training.")\n    digest = hashlib.sha256()\n    with source.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    if digest.hexdigest() != audit["sha256"]:\n        raise ValueError("CSV SHA-256 disagrees with audit")\n    args.out.mkdir(parents=True, exist_ok=True)\n    prof_path = args.out / "profile.json"\n    prof = json.loads(prof_path.read_text(encoding="utf-8")) if prof_path.exists() else profile(source, audit, args.out)\n    if prof["source_sha256"] != audit["sha256"]:\n        raise ValueError("Profile SHA-256 disagrees with audit")\n    if not args.profile_only:\n        train(source, audit, prof, args.out)\n\n\nif __name__ == "__main__":\n    main()\n', 'audit_paysim.py': '"""Read-only PaySim CSV audit. Python 3.10+, standard library only.\n\nDoes not clean, rewrite, sample, train on, or publish the input dataset.\nExact duplicate detection is deliberately reported as not performed.\n"""\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport sys\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\nFIELDS = [\n    "step", "type", "amount", "nameOrig", "oldbalanceOrg",\n    "newbalanceOrig", "nameDest", "oldbalanceDest", "newbalanceDest",\n    "isFraud", "isFlaggedFraud",\n]\nTYPES = {"CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"}\nBALANCES = ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]\n\n\ndef scan_csv(stream):\n    reader = csv.DictReader(stream)\n    if reader.fieldnames != FIELDS:\n        raise ValueError(\n            f"Expected exact PaySim header/order: {FIELDS}; got: {reader.fieldnames}. "\n            "Check file/version/encoding; do not silently rename columns."\n        )\n    missing = Counter({field: 0 for field in FIELDS})\n    invalid_numeric = Counter()\n    warnings = Counter()\n    essential_errors = Counter()\n    types = Counter()\n    labels = Counter()\n    by_type_label = Counter()\n    by_step = defaultdict(Counter)\n    examples = []\n    rows = 0\n    previous_step = None\n    amount_min = None\n    amount_max = None\n\n    def issue(row_num, field, reason):\n        if len(examples) < 12:\n            examples.append({"data_row": row_num, "field": field, "reason": reason})\n\n    for row_num, row in enumerate(reader, start=1):\n        rows += 1\n        if None in row or any(row.get(field) is None for field in FIELDS):\n            essential_errors["wrong_number_of_fields"] += 1\n            issue(row_num, "row", "wrong_number_of_fields")\n            continue\n        for field in FIELDS:\n            if row[field].strip().lower() in {"", "null", "none", "nan", "na", "n/a"}:\n                missing[field] += 1\n        for field in ["type", "nameOrig", "nameDest"]:\n            if not row[field].strip():\n                essential_errors[f"empty_{field}"] += 1\n                issue(row_num, field, "empty_essential_field")\n        if row["type"] not in TYPES:\n            essential_errors["unknown_type"] += 1\n            issue(row_num, "type", "unknown_or_noncanonical_type")\n        types[row["type"]] += 1\n        for field in ["nameOrig", "nameDest"]:\n            if row[field] != row[field].strip():\n                warnings[f"outer_whitespace_{field}"] += 1\n\n        nums = {}\n        for field in ["step", "amount", *BALANCES, "isFraud", "isFlaggedFraud"]:\n            try:\n                val = float(row[field])\n                if not math.isfinite(val):\n                    raise ValueError("nonfinite")\n                nums[field] = val\n            except (TypeError, ValueError, OverflowError):\n                invalid_numeric[field] += 1\n                issue(row_num, field, "not_a_finite_number")\n                if field in {"step", "amount", "isFraud"}:\n                    essential_errors[f"invalid_{field}"] += 1\n\n        step = nums.get("step")\n        if step is not None and (step < 1 or not step.is_integer()):\n            essential_errors["invalid_step_range_or_fraction"] += 1\n            issue(row_num, "step", "must_be_positive_integer")\n            step = None\n        if step is not None:\n            step = int(step)\n            if previous_step is not None and step < previous_step:\n                warnings["step_decreases_in_file_order"] += 1\n            previous_step = step\n            by_step[step]["rows"] += 1\n\n        amount = nums.get("amount")\n        if amount is not None:\n            amount_min = amount if amount_min is None else min(amount_min, amount)\n            amount_max = amount if amount_max is None else max(amount_max, amount)\n            if amount < 0:\n                essential_errors["negative_amount"] += 1\n                issue(row_num, "amount", "negative_amount_requires_review")\n            elif amount == 0:\n                warnings["zero_amount_keep_pending_business_review"] += 1\n        label = nums.get("isFraud")\n        if label is not None and label not in {0, 1}:\n            essential_errors["nonbinary_isFraud"] += 1\n            issue(row_num, "isFraud", "must_be_zero_or_one")\n        elif label is not None:\n            label = int(label)\n            labels[str(label)] += 1\n            by_type_label[(row["type"], label)] += 1\n            if step is not None:\n                by_step[step][f"label_{label}"] += 1\n        flag = nums.get("isFlaggedFraud")\n        if flag is not None and flag not in {0, 1}:\n            warnings["nonbinary_isFlaggedFraud"] += 1\n        for field in BALANCES:\n            if field in nums and nums[field] == 0:\n                warnings[f"zero_{field}_not_automatically_missing"] += 1\n            if field in nums and nums[field] < 0:\n                warnings[f"negative_{field}_review_only"] += 1\n\n    for field in ["nameOrig", "nameDest"]:\n        if missing[field]:\n            essential_errors[f"missing_token_{field}"] = missing[field]\n    steps = sorted(by_step)\n    splits = {}\n    if len(steps) >= 3:\n        n_train = min(len(steps) - 2, max(1, int(0.6 * len(steps))))\n        n_val = min(len(steps) - n_train - 1, max(1, int(0.2 * len(steps))))\n        ranges = {\n            "train": steps[:n_train],\n            "validation": steps[n_train:n_train + n_val],\n            "test": steps[n_train + n_val:],\n        }\n        for split_name, selected in ranges.items():\n            splits[split_name] = {\n                "min_step": selected[0], "max_step": selected[-1],\n                "observed_steps": len(selected),\n                "rows": sum(by_step[s]["rows"] for s in selected),\n                "fraud": sum(by_step[s]["label_1"] for s in selected),\n                "legitimate": sum(by_step[s]["label_0"] for s in selected),\n            }\n    blockers = bool(essential_errors) or rows == 0 or not splits\n    both_classes = all(x["fraud"] > 0 and x["legitimate"] > 0 for x in splits.values()) if splits else False\n    return {\n        "rows": rows,\n        "columns": FIELDS,\n        "missing_or_null_tokens_by_column": dict(missing),\n        "invalid_numeric_by_column": dict(invalid_numeric),\n        "essential_errors": dict(essential_errors),\n        "warnings_are_not_deletion_rules": dict(warnings),\n        "label_counts": dict(labels),\n        "fraud_rate": labels["1"] / rows if rows else None,\n        "type_counts": dict(types),\n        "type_label_counts": [\n            {"type": t, "label": label, "rows": count}\n            for (t, label), count in sorted(by_type_label.items())\n        ],\n        "step_min": steps[0] if steps else None,\n        "step_max": steps[-1] if steps else None,\n        "number_of_observed_steps": len(steps),\n        "amount_min": amount_min, "amount_max": amount_max,\n        "proposed_split_by_60_20_20_percent_of_observed_hours_not_rows": splits,\n        "both_classes_in_every_proposed_split": both_classes,\n        "sample_issues": examples,\n        "exact_duplicates": "NOT_CHECKED; no rows removed; repeated account IDs are not duplicates",\n        "feature_policy": {\n            "target": "isFraud",\n            "exclude_from_main_predictors": [*BALANCES, "isFlaggedFraud"],\n            "never_use_as_predictors": ["row_number", "event_id", "isFraud"],\n        },\n        "next_action": (\n            "Proceed to chronological baseline; review warnings and record duplicate policy. "\n            "This is a schema/quality audit, not a guarantee of no leakage or simulator artifacts."\n            if not blockers and both_classes else\n            "Review essential errors and/or chronological split before training. Do not auto-clean."\n        ),\n        "input_was_modified": False,\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("csv_path", type=Path)\n    parser.add_argument("--report", type=Path, help="New JSON path; existing files are never overwritten")\n    args = parser.parse_args()\n    source = args.csv_path.resolve(strict=True)\n    if not source.is_file():\n        parser.error("csv_path must be a file")\n    if args.report and args.report.exists():\n        parser.error("Report exists. Choose a new output filename.")\n    digest = hashlib.sha256()\n    with source.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    with source.open("r", encoding="utf-8-sig", newline="") as stream:\n        report = scan_csv(stream)\n    report["source"] = str(source)\n    report["bytes"] = source.stat().st_size\n    report["sha256"] = digest.hexdigest()\n    text = json.dumps(report, ensure_ascii=True, indent=2)\n    if args.report:\n        with args.report.open("x", encoding="utf-8") as stream:\n            stream.write(text + "\\n")\n        print(f"Report saved: {args.report.resolve()}")\n    else:\n        print(text)\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except (OSError, ValueError, csv.Error) as error:\n        print(f"Audit failed: {error}", file=sys.stderr)\n        sys.exit(2)\n'}
for name,content in FILES.items():(ROOT/'src'/name).write_text(content,encoding='utf8')
spec=importlib.util.spec_from_file_location('paysim_experiment',ROOT/'src/paysim_experiment.py')
exp=importlib.util.module_from_spec(spec);spec.loader.exec_module(exp)
def run(args,log):
    with (ROOT/log).open('w',encoding='utf8') as f:
        result=subprocess.run([sys.executable,*map(str,args)],stdout=f,stderr=subprocess.STDOUT)
    print((ROOT/log).read_text(encoding='utf8')[-6000:])
    if result.returncode:raise RuntimeError(f'Job failed, see {log}')


## 2 Find and verify source
Neu co nhieu ban CSV PaySim (nguon, raw, sample), phai chon SOURCE_OVERRIDE toi ban full. Snapshot phai khop manifest checkpoint.

In [ ]:
if SOURCE_OVERRIDE:
    SOURCE=Path(SOURCE_OVERRIDE)
else:
    matches=[]
    for p in Path('/kaggle/input').rglob('*.csv'):
        try:
            with p.open(encoding='utf-8-sig',newline='') as f:
                if next(csv.reader(f),None)==exp.FIELDS:matches.append(p)
        except (OSError,UnicodeError,csv.Error):pass
    if len(matches)!=1:raise RuntimeError(f'Found {len(matches)} PaySim CSVs. Add Input or set SOURCE_OVERRIDE to FULL raw CSV.')
    SOURCE=matches[0]
h=hashlib.sha256()
with SOURCE.open('rb') as f:
    for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
SOURCE_HASH=h.hexdigest()
if EXPECTED_SHA256 and SOURCE_HASH!=EXPECTED_SHA256:raise ValueError('Snapshot differs from checkpoint')
(ROOT/'source.json').write_text(json.dumps(dict(path=str(SOURCE),sha256=SOURCE_HASH,bytes=SOURCE.stat().st_size,
    source_url='https://www.kaggle.com/datasets/ealaxi/paysim1',checkpoint_hash_checked=bool(EXPECTED_SHA256)),indent=2))
print(SOURCE,SOURCE_HASH)


## 3 Audit and profile
Doc nguyen raw, khong xoa dong, khong thay so du 0 thanh missing. Audit dung Python chuan, co the mat vai phut. Profile chon moc gan 60/20/20 theo SO HANG, giu tron gio. Su dung split trong profile.json cho ML; split thoi gian so bo trong audit la tham khao khac, khong tron hai cach chia.

In [ ]:
AUDIT=ROOT/'audit.json'
if not AUDIT.exists():run([ROOT/'src/audit_paysim.py',SOURCE,'--report',AUDIT],'audit.log')
audit=json.loads(AUDIT.read_text(encoding='utf8'))
if audit['sha256']!=SOURCE_HASH or audit['essential_errors']:raise ValueError('Audit hash/errors need review')
if not (MODEL_OUT/'profile.json').exists():
    run([ROOT/'src/paysim_experiment.py','--csv',SOURCE,'--audit',AUDIT,'--out',MODEL_OUT,'--profile-only'],'profile.log')
prof=json.loads((MODEL_OUT/'profile.json').read_text(encoding='utf8'))
if prof['source_sha256']!=SOURCE_HASH:raise ValueError('Profile belongs to another snapshot')
print(json.dumps(prof['splits'],indent=2))


## 4 Train and select using validation
Baseline dung log1p(amount), type va chu ky gio tu step; khong dung ID, nhan, co luat hay so du. Bo ca so du truoc giao dich de don gian baseline, khong co nghia moi so du deu ro ri. Thu LR weight 1/10/50 va RF 50 cay depth 8. Ngan sach 1% canh bao validation la lua chon thi nghiem cua nhom. AP la average precision, khong goi nham thanh tich phan PR-AUC. Xem warnings hoi tu truoc khi khoa.

In [ ]:
exec('def fit_validation(source, audit, prof, out):\n    from sklearn.ensemble import RandomForestClassifier\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    import joblib\n    from threadpoolctl import threadpool_limits\n    if (out / "baseline_results.json").exists():\n        raise FileExistsError("baseline_results.json exists; use new output directory")\n    log("Loading only permitted columns plus isolated labels/rule baseline")\n    frame = pd.read_csv(source, usecols=["step", "type", "amount", "isFraud", "isFlaggedFraud"], dtype=DTYPES)\n    masks = {name: frame["step"].between(spec["min_step"], spec["max_step"]).to_numpy()\n             for name, spec in prof["splits"].items()}\n    categories = sorted(frame.loc[masks["train"], "type"].unique().tolist())\n    feature_names = ["log1p_amount", "hour_sin", "hour_cos", *[f"type_{c}" for c in categories]]\n    assert not any(c in feature_names for c in ["isFraud", "isFlaggedFraud", "step", "nameOrig", "nameDest"])\n    train_x = make_features(frame.loc[masks["train"]], categories)\n    val_x = make_features(frame.loc[masks["validation"]], categories)\n    train_y = frame.loc[masks["train"], "isFraud"].to_numpy()\n    val_y = frame.loc[masks["validation"], "isFraud"].to_numpy()\n    candidates = [\n        (f"LR_weight_{weight}", make_pipeline(StandardScaler(), LogisticRegression(\n            C=1.0, max_iter=250, class_weight={0: 1.0, 1: float(weight)}, random_state=42)))\n        for weight in (1, 10, 50)\n    ]\n    candidates.append(("RF_50_depth8_balanced", RandomForestClassifier(\n        n_estimators=50, max_depth=8, min_samples_leaf=20, max_samples=0.5,\n        class_weight="balanced_subsample", n_jobs=2, random_state=42)))\n    summaries = []\n    best = None\n    best_key = None\n    with threadpool_limits(limits=2):\n        for name, model in candidates:\n            log(f"Fit {name}: {len(train_y):,} rows, ALL chronological train rows")\n            started = time.perf_counter()\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter("always")\n                model.fit(train_x, train_y)\n            fit_seconds = time.perf_counter() - started\n            score = model.predict_proba(val_x)[:, 1]\n            threshold = select_threshold(val_y, score)\n            evaluation = metrics(val_y, score, threshold)\n            result = {"model": name, "fit_seconds": fit_seconds,\n                      "warnings": [str(w.message) for w in caught], "validation": evaluation}\n            summaries.append(result)\n            log(f"Validation {name}: AP={evaluation[\'average_precision\']:.6f}, recall={evaluation[\'recall\']:.4f}, alerts={evaluation[\'alert_rate\']:.4%}")\n            key = (evaluation["recall"], evaluation["precision"], evaluation["average_precision"])\n            if best_key is None or key > best_key:\n                best_key, best = key, (name, model, threshold)\n        del train_x, val_x\n        selected_name, selected, threshold = best\n        log(f"LOCKED selection before test scoring: {selected_name}, threshold={threshold}")\n        return dict(selected_name=selected_name,selected=selected,threshold=threshold,\n                    categories=categories,frame=frame,masks=masks,summaries=summaries,\n                    versions={p:importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'scipy\',\'joblib\',\'threadpoolctl\']})\n',exp.__dict__)
state=exp.fit_validation(SOURCE,audit,prof,MODEL_OUT)
import joblib,numpy as np,pandas as pd
joblib.dump(state['selected'],MODEL_OUT/'selected_model.joblib')
lock=dict(model=state['selected_name'],threshold=float(state['threshold']),
    type_categories=state['categories'],source_sha256=SOURCE_HASH,
    feature_version='safe_v1_logamount_hour_type',
    features=['log1p_amount','hour_sin','hour_cos']+['type_'+c for c in state['categories']],
    policy='maximize validation recall with <=1% alerts; then precision and AP; tied scores kept together',
    splits=prof['splits'],versions=state['versions'])
(MODEL_OUT/'model_manifest.json').write_text(json.dumps(lock,indent=2))
(MODEL_OUT/'validation_results.json').write_text(json.dumps(state['summaries'],indent=2))
pd.json_normalize(state['summaries']).to_csv(MODEL_OUT/'validation_metrics.csv',index=False)
check_frame=state['frame'].loc[state['masks']['validation']].head(64)
check_x=exp.make_features(check_frame,state['categories'])
np.testing.assert_allclose(joblib.load(MODEL_OUT/'selected_model.joblib').predict_proba(check_x),state['selected'].predict_proba(check_x))
(MODEL_OUT/'reload_check.json').write_text(json.dumps(dict(status='PASS',rows=len(check_frame),split='validation')))
print('MODEL LOCKED before test:',lock)


## 5 Final test after design is frozen
Ngay 6 doi RUN_FINAL_TEST=True o dau, giu nguyen thiet ke duoc chon bang validation. Neu da xem test de chon feature/nguong thi day khong con la test doc lap. Timing theo lo chi la offline batch, khong phai latency p95 hay throughput Kafka/Spark. .joblib cua sklearn khong load truc tiep bang Spark MLlib.

In [ ]:
if not RUN_FINAL_TEST:
    print('TEST SKIPPED: review validation, freeze design, then enable RUN_FINAL_TEST.')
else:
    import time
    test=state['frame'].loc[state['masks']['test']]
    tx=exp.make_features(test,state['categories']);ty=test.isFraud.to_numpy()
    t=time.perf_counter();score=state['selected'].predict_proba(tx)[:,1];elapsed=time.perf_counter()-t
    result=exp.metrics(ty,score,state['threshold'])
    rule=((test.type=='TRANSFER')&(test.amount>200000)).to_numpy(dtype=float)
    bases={'always_legitimate':exp.metrics(ty,np.zeros(len(ty)),.5),
           'fixed_transfer_gt_200000':exp.metrics(ty,rule,.5),
           'dataset_flag_rule':exp.metrics(ty,test.isFlaggedFraud.to_numpy(dtype=float),.5)}
    (MODEL_OUT/'test_metrics.json').write_text(json.dumps(dict(selected=state['selected_name'],metrics=result,
        baselines=bases,offline_bulk_predict_seconds=elapsed,
        warning='Offline batch timing only; validation alert budget may not hold on test under drift'),indent=2))
    pd.DataFrame([dict(model=state['selected_name'],**result)]+[dict(model=k,**v) for k,v in bases.items()]).to_csv(MODEL_OUT/'test_metrics.csv',index=False)
    import matplotlib.pyplot as plt
    from sklearn.metrics import PrecisionRecallDisplay,ConfusionMatrixDisplay
    fig,ax=plt.subplots(figsize=(6,5));PrecisionRecallDisplay.from_predictions(ty,score,ax=ax,name=state['selected_name'])
    fig.tight_layout();fig.savefig(MODEL_OUT/'test_pr_curve.png',dpi=170);plt.show();plt.close(fig)
    fig,ax=plt.subplots(figsize=(5,5));ConfusionMatrixDisplay.from_predictions(ty,score>=state['threshold'],ax=ax,colorbar=False)
    fig.tight_layout();fig.savefig(MODEL_OUT/'test_confusion_matrix.png',dpi=170);plt.show();plt.close(fig)
    demo=test[['step','type','amount']].head(20).copy();demo['score']=score[:len(demo)]
    demo['threshold']=state['threshold'];demo['predicted_fraud']=demo.score>=state['threshold']
    demo.to_csv(MODEL_OUT/'prediction_examples.csv',index=False)
    print(json.dumps(result,indent=2))


## 6 Export artifacts
Save Version > Save & Run All, cho version hoan tat, kiem Output va tai ZIP. Cac file source va model_manifest can mang sang phan inference. Co the dung sklearn predict_proba theo batch trong foreachBatch voi cung feature code; hoac train lai bang MLlib. Notebook nay chua trien khai Kafka/Spark serving. Khong so sanh baseline duoc chay khac snapshot/split nhu cung mot thi nghiem.

In [ ]:
freeze=subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True)
(ROOT/'environment_freeze.txt').write_text(freeze.stdout,encoding='utf8')
versions=dict(state['versions'],python=platform.python_version())
(ROOT/'environment.json').write_text(json.dumps(versions,indent=2))
(ROOT/'requirements-lock.txt').write_text('\n'.join(f'{k}=={v}' for k,v in versions.items() if k!='python')+'\n')
archive=ROOT.parent/(EXPERIMENT+'_artifacts.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(ROOT))
print('DOWNLOAD',archive)
print('Final test present:',(MODEL_OUT/'test_metrics.json').exists())
